# nexa-backtest Walkthrough

**A beginner's guide to backtesting a European power trading algorithm**

---

This notebook walks you through `nexa-backtest` from first principles. No prior knowledge of trading, power markets, or backtesting is assumed.

By the end you will:
- Understand what a backtester does and why European power markets need their own
- Know what an MTU, DA auction, gate closure, and VWAP are
- Have written and run two trading algorithms against real fixture data
- Be able to read and interpret every metric in the results summary
- Understand the look-ahead bias trap and how `publication_offset` prevents it

**Estimated reading time: 20-30 minutes**

## 1. What Is a Backtester?

A **backtester** is a system that takes a trading algorithm and replays historical market data through it, as if the algorithm were trading in real-time. Instead of waiting months to find out whether a strategy works, you can test it against years of history in seconds.

Here is the basic idea:

```
Historical market data
        │
        ▼
┌───────────────────┐
│   Backtest Engine │   ← replays data event by event
│                   │
│   Simulated Clock │   ← time advances to each auction
│   Matching Engine │   ← fills orders using real clearing prices
└───────────────────┘
        │
        ▼
   Your Algorithm
   (on_auction_open, on_fill, etc.)
        │
        ▼
   Results: fills, PnL, VWAP alpha
```

The engine **guarantees** that your algorithm only ever sees data that would have been available at that simulated point in time. No peeking into the future. That guarantee is what makes backtest results trustworthy.

## 2. Why European Power Markets Need Their Own Backtester

Most open-source backtesting tools were built for equities or crypto. They assume:
- You can buy or sell at any time during the trading day
- Prices are continuous (no auctions with a single clearing price)
- Settlement is in shares or coins, not megawatt-hours of electricity

European power markets work very differently:

| Feature | Equities | EU Power Markets |
|---|---|---|
| Market structure | Continuous order book | Auctions + continuous intraday |
| Delivery | Immediate | Future 15-minute or hourly periods |
| Settlement | Cash or shares | Physical electricity (MWh) |
| Time resolution | Tick data | 15-minute MTUs (since Sept 2025) |
| Gate closure | N/A | Hard deadline: bids must be in before noon D-1 |
| Benchmark | Mid-price, TWAP | VWAP of clearing prices |

`nexa-backtest` is purpose-built for these markets. It handles the auction lifecycle, 15-minute MTU resolution, and VWAP benchmarking natively.

## 3. Core Concepts

Before writing any code, let's make sure the key terms are clear.

### MTU — Market Time Unit

An MTU is a 15-minute block of time that represents one **delivery period** in European power markets. When you trade electricity, you are buying or selling the right to consume or generate power during a specific 15-minute window.

```
00:00 ─── 00:15 ─── 00:30 ─── 00:45 ─── 01:00 ...
  │ MTU 1  │ MTU 2  │ MTU 3  │ MTU 4  │
```

EU power markets transitioned to 15-minute MTUs on 30 September 2025. Before that, the resolution was hourly. This backtester handles both.

### DA Auction — Day-Ahead Auction

In the Day-Ahead market, participants submit buy and sell bids for each MTU of the **next calendar day** before a hard deadline called the **gate closure** (noon on the day before delivery, for Nord Pool).

After gate closure, the exchange runs an auction algorithm that finds a single **clearing price** per MTU — the price where supply meets demand.

```
Today (D-1)                    Tomorrow (D)
   │                               │
   ▼                               ▼
12:00 UTC ← Gate closure      00:00 – 23:45 UTC: delivery periods
Submit bids before this!       96 MTUs, each with its own clearing price
```

### Price-Taker Assumption

When backtesting, we assume that the algo's orders are **too small to move the market**. The clearing price comes from historical data, and the algo fills at that price if its bid is good enough. This is realistic for most participants.

### Gate Closure

Gate closure is the **hard deadline** for submitting or modifying orders for a given delivery period. In the backtest engine, the simulated clock is set to D-1 12:00 UTC when processing each delivery day's auction. After this point, orders are matched and fills are issued.

### VWAP — Volume-Weighted Average Price

VWAP is the **benchmark**. It is the weighted average of all DA clearing prices across the backtest period, weighted by how much volume was traded at each price:

```
VWAP = Σ(price × volume) / Σ(volume)
```

If your algo's average buy price is **below** VWAP, it bought electricity cheaper than the market average — that is good. If your algo's average sell price is **above** VWAP, it sold above average — also good.

### Alpha

Alpha is the difference between your execution quality and the VWAP benchmark. Positive total alpha in EUR means your algo's timing added value compared to simply participating at every period at the clearing price.

## 4. Exploring the Fixture Data

Let's start by loading and understanding the historical DA clearing price data that ships with this repo. All backtests run against Parquet files in `tests/fixtures/`.

In [ ]:
import pandas as pd
from pathlib import Path

# The fixture data lives one directory up from this notebook
FIXTURES_DIR = Path("../tests/fixtures")

# Load the DA clearing prices Parquet file
da_prices = pd.read_parquet(FIXTURES_DIR / "da_prices.parquet")

print(f"Shape: {da_prices.shape}")
print(f"\nColumns: {list(da_prices.columns)}")
print(f"\nDate range: {da_prices['timestamp'].min()} to {da_prices['timestamp'].max()}")
print(f"\nZones: {da_prices['zone'].unique()}")
print(f"\nPrice range: {da_prices['price_eur_mwh'].min():.2f} – {da_prices['price_eur_mwh'].max():.2f} EUR/MWh")
print(f"\nVolume range: {da_prices['volume_mwh'].min():.1f} – {da_prices['volume_mwh'].max():.1f} MWh")
print(f"\nFirst few rows:")
da_prices.head(8)

Each row represents one MTU — one 15-minute delivery period. With 96 MTUs per day and 31 days in March 2026, we have 96 × 31 = 2,976 rows.

The `product_id` column (added by the data loader) identifies which delivery period each price applies to. The `price_eur_mwh` column is the historical clearing price for that period — the price everyone who bid above it paid (for buys), or received (for sells).

Let's visualise the price time series and the average daily profile to get a feel for the data.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebooks without a display
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# --- Panel 1: Full price time series ---
ax1 = axes[0]
ax1.plot(da_prices["timestamp"], da_prices["price_eur_mwh"],
         linewidth=0.6, color="#2563eb", alpha=0.8)
ax1.set_title("NO1 DA Clearing Prices — March 2026 (all 2,976 MTUs)", fontsize=13)
ax1.set_ylabel("EUR/MWh")
ax1.set_xlabel("")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax1.grid(True, alpha=0.3)

# Draw the market VWAP as a horizontal reference line
market_vwap = (da_prices["price_eur_mwh"] * da_prices["volume_mwh"]).sum() / da_prices["volume_mwh"].sum()
ax1.axhline(market_vwap, color="#dc2626", linewidth=1.2, linestyle="--", label=f"Market VWAP = {market_vwap:.2f} EUR/MWh")
ax1.legend(fontsize=10)

# --- Panel 2: Average intraday price profile (the "daily shape") ---
ax2 = axes[1]
# Extract the time-of-day from UTC timestamps for grouping
da_prices["time_of_day"] = da_prices["timestamp"].dt.strftime("%H:%M")
daily_profile = da_prices.groupby("time_of_day")["price_eur_mwh"].mean()

# Plot with hour labels on x-axis
ax2.bar(range(len(daily_profile)), daily_profile.values, color="#2563eb", alpha=0.7, width=0.9)
ax2.set_title("Average DA Clearing Price by Time of Day (March 2026 average)", fontsize=13)
ax2.set_ylabel("EUR/MWh")
ax2.set_xlabel("Time of Day (UTC)")
# Label every 4th MTU (= every hour)
tick_positions = list(range(0, 96, 4))
tick_labels = [daily_profile.index[i] for i in tick_positions]
ax2.set_xticks(tick_positions)
ax2.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=8)
ax2.axhline(daily_profile.mean(), color="#dc2626", linewidth=1.2, linestyle="--", label="Daily average")
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("da_prices_overview.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\nMarket VWAP across March 2026: {market_vwap:.4f} EUR/MWh")
print("Notice the sinusoidal pattern — prices peak in the morning and evening, dip overnight.")

## 5. The Algorithm Interface: SimpleAlgo Lifecycle

`SimpleAlgo` is the base class for all hook-based trading algorithms. You subclass it and override only the hooks you need. The engine calls your hooks in a well-defined order.

Here is the full lifecycle for a DA backtest:

```
Engine starts
     │
     ▼
on_setup(ctx)                   ← called once at the start
     │                            subscribe to signals, initialise state
     │
     ▼  (loop: for each delivery day)
on_signal(ctx, name, value)     ← called once per day per subscribed signal
     │                            receive the latest forecast value
     │
     ▼  (loop: for each MTU in the day, 96 times)
on_auction_open(ctx, auction)   ← called once per MTU
     │                            place orders here via ctx.place_order(...)
     │
     │  (engine matches your orders against clearing price)
     │
     ▼  (for each filled order)
on_fill(ctx, fill)              ← called when an order is filled
     │                            record the fill, update internal state
     │
     ▼  (next delivery day...)
     │
     ▼
on_teardown(ctx)                ← called once at the end
                                  log positions, print summary, cleanup
```

**The TradingContext (`ctx`) is the only thing your algo talks to.** It provides:
- `ctx.now()` — the current simulated time
- `ctx.place_order(order)` — submit a buy or sell order
- `ctx.get_signal(name)` — read the latest signal value
- `ctx.get_position(product_id)` — check your current position
- `ctx.log(message)` — emit a log message tagged with simulated time

Your algo never imports anything from the engine. This is the key design rule: the same code runs unchanged under the backtest, paper, and live engines.

## 6. Your Simplest Algo: Always Buy

The simplest possible algorithm buys every single DA period at a price so high it always fills. This gives us a baseline: if you had bought every MTU at the clearing price all month, what would your results look like?

This is also a useful sanity check. If the backtester is working correctly, an always-buy algo's average price should equal the market VWAP (since it participates at every single clearing price).

In [ ]:
from nexa_backtest import SimpleAlgo, Order
from nexa_backtest.context import TradingContext
from nexa_backtest.types import AuctionInfo, Fill


class AlwaysBuyAlgo(SimpleAlgo):
    """The simplest possible algo: buy every MTU at a price that always fills.

    We bid 999 EUR/MWh, which is far above any realistic clearing price,
    so every order fills. The fill price is always the clearing price (not
    our bid) because of the price-taker / uniform price auction rule.
    """

    def on_setup(self, ctx: TradingContext) -> None:
        # Initialise a counter so we can report how many fills we got
        self.fill_count = 0
        ctx.log("AlwaysBuyAlgo initialised")

    def on_auction_open(self, ctx: TradingContext, auction: AuctionInfo) -> None:
        # For every MTU auction that opens, place a buy order at 999 EUR/MWh.
        # Because the actual clearing prices are ~20-80 EUR/MWh, this will
        # always satisfy the fill condition: bid_price (999) >= clearing_price.
        order = Order.buy(
            product_id=auction.product_id,
            volume_mw=1.0,      # buy 1 MW per MTU
            price_eur_mwh=999,  # intentionally far above market — guarantees a fill
        )
        ctx.place_order(order)

    def on_fill(self, ctx: TradingContext, fill: Fill) -> None:
        # Called after each fill — we just count them
        self.fill_count += 1

    def on_teardown(self, ctx: TradingContext) -> None:
        ctx.log(f"Backtest complete. Total fills: {self.fill_count}")


print("AlwaysBuyAlgo defined successfully.")
print("Note: we never import BacktestEngine inside the algo — it stays engine-agnostic.")

## 7. Understanding DA Auction Matching

Before running the backtest, let's understand exactly how the engine decides whether to fill an order.

The `DAAuctionMatcher` implements the **price-taker, uniform-price auction** rule:

| Order type | Condition for fill |
|---|---|
| Buy limit order | `bid_price >= clearing_price` |
| Sell limit order | `bid_price <= clearing_price` |
| Market order | Always filled |

When filled, the execution price is **always the clearing price** — not your bid. This is how real DA auctions work: you state the maximum you are willing to pay, and if the market clears at a lower price, you pay the lower price.

Let's see this in action directly with the matching engine:

In [ ]:
from decimal import Decimal
from datetime import datetime, timezone
from nexa_backtest.engines.matching import DAAuctionMatcher
from nexa_backtest.types import Order

# Historical clearing price for this hypothetical MTU: 50 EUR/MWh
clearing_price = Decimal("50.00")
fill_time = datetime(2026, 3, 1, 13, 0, tzinfo=timezone.utc)
matcher = DAAuctionMatcher(clearing_price=clearing_price, fill_timestamp=fill_time)

print("Historical clearing price: 50.00 EUR/MWh")
print("=" * 55)

# --- Case 1: Buy bid above clearing price → fills at clearing price ---
order_above = Order.buy("NO1-QH-0000", volume_mw=10, price_eur_mwh=55)
result1 = matcher.match(order_above)
print(f"\nCase 1: BUY bid=55, clearing=50")
print(f"  Status: {result1.status}")
if result1.fill:
    print(f"  Fill price: {result1.fill.price} EUR/MWh  ← clearing price, not bid!")
    print(f"  Fill volume: {result1.fill.volume} MW")

# --- Case 2: Buy bid below clearing price → rejected ---
order_below = Order.buy("NO1-QH-0000", volume_mw=10, price_eur_mwh=45)
result2 = matcher.match(order_below)
print(f"\nCase 2: BUY bid=45, clearing=50")
print(f"  Status: {result2.status}")
print(f"  Reason: {result2.rejection_reason}")

# --- Case 3: Buy bid exactly at clearing price → fills ---
order_exact = Order.buy("NO1-QH-0000", volume_mw=10, price_eur_mwh=50)
result3 = matcher.match(order_exact)
print(f"\nCase 3: BUY bid=50, clearing=50  (bid exactly equals clearing)")
print(f"  Status: {result3.status}")
if result3.fill:
    print(f"  Fill price: {result3.fill.price} EUR/MWh")

# --- Case 4: Sell bid below clearing price → fills at clearing price ---
order_sell = Order.sell("NO1-QH-0000", volume_mw=5, price_eur_mwh=45)
result4 = matcher.match(order_sell)
print(f"\nCase 4: SELL bid=45, clearing=50  (willing to sell for 45, market pays 50)")
print(f"  Status: {result4.status}")
if result4.fill:
    print(f"  Fill price: {result4.fill.price} EUR/MWh  ← you get the clearing price!")

print("\n" + "=" * 55)
print("Key takeaway: you never pay more than clearing (buy) or receive")
print("less than clearing (sell). Your bid is just the accept/reject threshold.")

## 8. Running the Backtest

Now let's put it all together. We create a `BacktestEngine`, configure it with our algo and data, and call `.run()`.

The engine:
1. Loads DA clearing prices from the Parquet file
2. Advances the simulated clock to D-1 12:00 UTC for each delivery day
3. Calls `on_auction_open` for each of the 96 MTUs in that day
4. Matches all placed orders against clearing prices
5. Calls `on_fill` for each filled order
6. Returns a `BacktestResult` with all fills and PnL metrics

In [ ]:
from datetime import date
from decimal import Decimal
from pathlib import Path
from nexa_backtest import BacktestEngine

# Create and run the backtest
engine = BacktestEngine(
    algo=AlwaysBuyAlgo(),           # our always-buy strategy
    exchange="nordpool",             # exchange identifier
    start=date(2026, 3, 1),         # first delivery day (inclusive)
    end=date(2026, 3, 31),          # last delivery day (inclusive)
    products=["NO1_DA"],            # NO1 zone, Day-Ahead market
    data_dir=Path("../tests/fixtures"),  # where da_prices.parquet lives
    capital=Decimal("100000"),      # starting capital in EUR (informational)
)

print("Running backtest...")
result_always_buy = engine.run()
print("Done!\n")

# Print the formatted summary
print(result_always_buy.summary())

## 9. Understanding Every Result Metric

The summary output contains several metrics. Let's inspect them one by one and understand what each means.

### Accessing raw result fields

The `BacktestResult` object gives you access to all the underlying data programmatically:

In [ ]:
r = result_always_buy
p = r.pnl  # PnlSummary object

print("=== BacktestResult fields ===\n")

print(f"algo_name:   {r.algo_name}")
print(f"exchange:    {r.exchange}")
print(f"start:       {r.start}")
print(f"end:         {r.end}")
print(f"total fills: {len(r.fills)}")

print("\n=== PnlSummary fields ===\n")

print(f"market_vwap:       {p.market_vwap:.4f} EUR/MWh")
print(f"  ↳ Volume-weighted average of ALL 2,976 clearing prices in the period.")
print(f"  ↳ This is the benchmark everything else is measured against.\n")

print(f"buys.count:        {p.buys.count}")
print(f"  ↳ Number of filled buy orders.\n")

print(f"buys.volume_mwh:   {p.buys.volume_mwh:.1f} MWh")
print(f"  ↳ Total volume bought. 1 MW × 0.25 h per MTU = 0.25 MWh each fill.")
print(f"  ↳ {p.buys.count} fills × 0.25 MWh = {float(p.buys.volume_mwh):.1f} MWh expected.\n")

print(f"buys.avg_price:    {p.buys.avg_price:.4f} EUR/MWh")
print(f"  ↳ Volume-weighted average of our fill prices.")
print(f"  ↳ For AlwaysBuyAlgo this should be very close to market_vwap.\n")

print(f"buys.vwap_alpha:   {p.buys.vwap_alpha:.4f} EUR/MWh")
print(f"  ↳ avg_price - market_vwap. For buys, NEGATIVE is good (bought below VWAP).")
print(f"  ↳ For AlwaysBuyAlgo this should be near zero (we buy everything).\n")

print(f"buys.total_alpha_eur: {p.buys.total_alpha_eur:.4f} EUR")
print(f"  ↳ vwap_alpha × volume_mwh in EUR. The total EUR value of execution quality.\n")

print(f"buys.win_rate:     {p.buys.win_rate:.1%}")
print(f"  ↳ Fraction of fills where fill_price < market_vwap (for buys).")
print(f"  ↳ 50% means you bought below VWAP half the time — random.\n")

print(f"total_alpha_eur:   {p.total_alpha_eur:.4f} EUR")
print(f"  ↳ buys.total_alpha_eur + sells.total_alpha_eur.")
print(f"  ↳ The single number that tells you whether the algo added value.")

## 10. VWAP Alpha Deep Dive

VWAP alpha is the most important metric for power market execution. Let's understand it in depth.

### What is the question VWAP alpha answers?

> "If I had participated at every single clearing price with equal volume (the naive baseline), would my algo have done better or worse?"

A passive market participant who buys 1 MW in every MTU would pay exactly the VWAP. Your algo's job is to beat that — buy in the cheap periods, skip the expensive ones.

### Alpha formula

```
For buys:
  vwap_alpha   = algo_avg_price - market_vwap
  (negative = good: you bought below the market average)

  total_alpha_eur = -vwap_alpha × volume_mwh
  (positive = good: you saved money vs passive participation)

For sells:
  vwap_alpha   = market_vwap - algo_avg_price
  (negative = good: you sold above the market average)

  total_alpha_eur = -vwap_alpha × volume_mwh
  (positive = good: you earned more than passive participation)
```

### Win rate

Win rate is the fraction of fills where your price beat VWAP:
- **Buy win**: fill price < market VWAP (you bought cheap)
- **Sell win**: fill price > market VWAP (you sold expensive)

A 50% win rate means you were no better than random. A good strategy should have win rate > 50%.

### Why VWAP and not a fixed benchmark?

In power markets, the VWAP changes every single month, season, and year. A fixed benchmark like "50 EUR/MWh" would be meaningless in a cold winter or a summer of cheap wind. VWAP self-calibrates to the actual market conditions during the test period.

Let's look at our fills to visualise where they sit relative to VWAP:

In [ ]:
import numpy as np

# Extract fill prices and timestamps into a DataFrame for plotting
fills_df = pd.DataFrame([
    {
        "timestamp": f.timestamp,
        "price": float(f.price),
        "volume": float(f.volume),
        "side": f.side,
        "product_id": f.product_id,
    }
    for f in result_always_buy.fills
])

market_vwap_f = float(result_always_buy.pnl.market_vwap)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: scatter of fill prices over time vs VWAP line ---
ax1 = axes[0]
below_vwap = fills_df[fills_df["price"] < market_vwap_f]
above_vwap = fills_df[fills_df["price"] >= market_vwap_f]

ax1.scatter(below_vwap["timestamp"], below_vwap["price"],
            s=4, color="#16a34a", alpha=0.5, label=f"Below VWAP ({len(below_vwap)} fills, good)")
ax1.scatter(above_vwap["timestamp"], above_vwap["price"],
            s=4, color="#dc2626", alpha=0.5, label=f"Above VWAP ({len(above_vwap)} fills)")
ax1.axhline(market_vwap_f, color="black", linewidth=1.5, linestyle="--",
            label=f"Market VWAP = {market_vwap_f:.2f} EUR/MWh")
ax1.set_title("Fill Prices vs Market VWAP (AlwaysBuyAlgo)", fontsize=12)
ax1.set_ylabel("Fill Price (EUR/MWh)")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Right panel: distribution of fill prices relative to VWAP ---
ax2 = axes[1]
price_vs_vwap = fills_df["price"] - market_vwap_f
ax2.hist(price_vs_vwap, bins=40, color="#2563eb", alpha=0.7, edgecolor="white")
ax2.axvline(0, color="#dc2626", linewidth=2, linestyle="--", label="VWAP (zero line)")
ax2.axvline(float(result_always_buy.pnl.buys.vwap_alpha), color="#f59e0b",
            linewidth=2, linestyle="-",
            label=f"Mean alpha = {float(result_always_buy.pnl.buys.vwap_alpha):.3f} EUR/MWh")
ax2.set_title("Distribution of Fill Price vs VWAP", fontsize=12)
ax2.set_xlabel("Fill Price - VWAP (EUR/MWh)")
ax2.set_ylabel("Number of Fills")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("fill_prices_vs_vwap.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"Win rate (bought below VWAP): {result_always_buy.pnl.buys.win_rate:.1%}")
print(f"For AlwaysBuyAlgo, win rate ~50% is expected — we buy everything, cheap and expensive.")

## 11. A Signal-Driven Algo

The `AlwaysBuyAlgo` has zero intelligence — it buys everything regardless of price. A real strategy uses market information to decide when to trade.

**Signals** are time-series data sources: price forecasts, weather data, load forecasts, gas prices. The signal system is the primary way to inject this information into your algo.

Here we use the `CsvSignalProvider` to load `price_forecast.csv`, which contains a forecast of the DA clearing price with slight noise added to the actual prices. In a real system this might come from an ML model, a weather service, or a third-party data vendor.

### How CsvSignalProvider works

```
CSV file:
  timestamp,value
  2026-03-01T00:00:00+00:00,42.3
  2026-03-01T00:15:00+00:00,38.7
  ...

CsvSignalProvider(
    name="price_forecast",
    path=...,
    unit="EUR/MWh",
    description="DA price forecast",
    publication_offset=timedelta(hours=36),  # forecasts published 36h before delivery
)
```

The `publication_offset` is crucial. A value with delivery timestamp T was published at `T - 36h`. At simulated time S (D-1 12:00 UTC), only values where `T <= S + 36h` are visible. This prevents look-ahead bias. We'll demonstrate this shortly.

### The ForecastAlgo strategy

The idea is simple: if the forecast price for a given MTU is **below the expected VWAP** (roughly estimated as the rolling mean of recent forecasts), then it is a cheap period — buy it. Otherwise, skip it.

In [ ]:
from datetime import timedelta
from nexa_backtest import SimpleAlgo, Order, CsvSignalProvider
from nexa_backtest.context import TradingContext, SignalValue
from nexa_backtest.types import AuctionInfo, Fill
from nexa_backtest.exceptions import SignalError


class ForecastAlgo(SimpleAlgo):
    """A signal-driven algo that buys MTUs forecasted to be below average price.

    Strategy:
    - Subscribe to a price forecast signal.
    - In on_signal, compute the rolling mean of the day's forecasted prices.
    - In on_auction_open, only buy if the forecast for this MTU is below
      the day's estimated average (i.e., it is expected to be a cheap period).
    """

    def on_setup(self, ctx: TradingContext) -> None:
        # Subscribe to the price forecast signal.
        # The engine will make this signal available before on_auction_open fires.
        self.subscribe_signal("price_forecast")
        self.daily_threshold: float = 50.0   # fallback if no signal yet
        self.fill_count = 0
        self.skipped_count = 0
        ctx.log("ForecastAlgo initialised — subscribed to price_forecast signal")

    def on_signal(self, ctx: TradingContext, name: str, value: SignalValue) -> None:
        # Called once per day before on_auction_open for that day.
        # Use the signal history to estimate today's average forecast price,
        # which we will use as the buy/skip threshold.
        if name == "price_forecast":
            # Retrieve the last 96 signal values visible right now (one full day)
            history = ctx.get_signal_history("price_forecast", lookback=96)
            if history:
                # Compute simple mean of recent forecasted prices as our threshold
                self.daily_threshold = sum(sv.value for sv in history) / len(history)
                ctx.log(
                    f"[{ctx.now().date()}] Forecast threshold updated to "
                    f"{self.daily_threshold:.2f} EUR/MWh from {len(history)} values"
                )

    def on_auction_open(self, ctx: TradingContext, auction: AuctionInfo) -> None:
        # Try to get the forecast for this specific MTU's delivery period.
        try:
            signal = ctx.get_signal("price_forecast")
        except SignalError:
            # No forecast available yet — skip this period rather than guess.
            self.skipped_count += 1
            return

        forecast_price = signal.value

        # Buy only if the forecast says this MTU is cheaper than average.
        # This is our "edge": skip expensive periods, focus on cheap ones.
        if forecast_price < self.daily_threshold:
            order = Order.buy(
                product_id=auction.product_id,
                volume_mw=1.0,
                # Bid the forecast price — we are willing to pay what we expect
                price_eur_mwh=forecast_price,
            )
            ctx.place_order(order)
        else:
            self.skipped_count += 1

    def on_fill(self, ctx: TradingContext, fill: Fill) -> None:
        self.fill_count += 1

    def on_teardown(self, ctx: TradingContext) -> None:
        ctx.log(
            f"Complete. fills={self.fill_count}, skipped={self.skipped_count}"
        )


print("ForecastAlgo defined.")
print("It will only buy MTUs where the forecast price is below the day's average forecast.")

Now let's run `ForecastAlgo` with the price forecast signal. We pass a `CsvSignalProvider` explicitly so we can control the `publication_offset`.

In [ ]:
from datetime import timedelta
from nexa_backtest import BacktestEngine, CsvSignalProvider

# Build the signal provider explicitly.
# publication_offset=timedelta(hours=36) means:
#   A forecast value with delivery timestamp T was published at T - 36h.
#   The auction clock is set to D-1 12:00 UTC.
#   For D=2026-03-02 (midnight), auction time = 2026-03-01 12:00 UTC.
#   36h window extends to: 2026-03-01 12:00 + 36h = 2026-03-03 00:00 UTC.
#   So ALL 96 delivery periods of 2026-03-02 (00:00..23:45) are visible. ✓
forecast_provider = CsvSignalProvider(
    name="price_forecast",
    path=Path("../tests/fixtures/signals/price_forecast.csv"),
    unit="EUR/MWh",
    description="Synthetic DA price forecast (actual + noise)",
    frequency=timedelta(minutes=15),
    publication_offset=timedelta(hours=36),
)

engine_forecast = BacktestEngine(
    algo=ForecastAlgo(),
    exchange="nordpool",
    start=date(2026, 3, 1),
    end=date(2026, 3, 31),
    products=["NO1_DA"],
    data_dir=Path("../tests/fixtures"),
    capital=Decimal("100000"),
    signals=[forecast_provider],  # pass the provider explicitly
)

print("Running ForecastAlgo backtest...")
result_forecast = engine_forecast.run()
print("Done!\n")
print(result_forecast.summary())

## 12. Look-Ahead Bias: The #1 Backtesting Mistake

> **Warning:** Look-ahead bias silently inflates your backtest performance. It is the most common mistake in backtesting and can make a completely useless strategy appear highly profitable. Always verify that your signals use a correct `publication_offset`.

### What is look-ahead bias?

Look-ahead bias happens when your algo uses data that it **could not have known** at the time the trading decision was made.

**Example without look-ahead bias (correct):**
- Auction clock: 2026-03-01 12:00 UTC (deciding trades for 2026-03-02)
- Forecast with `publication_offset=timedelta(hours=36)`:
  - Only forecasts for delivery periods up to `2026-03-01 12:00 + 36h = 2026-03-03 00:00` are visible
  - Forecasts for 2026-03-02 are available ✓

**Example with look-ahead bias (wrong):**
- If `publication_offset=timedelta(hours=0)` (no offset):
  - At auction time 2026-03-01 12:00, only forecasts with `timestamp <= 2026-03-01 12:00` are visible
  - The forecasts for 2026-03-02 (timestamps 2026-03-02 00:00..23:45) are NOT yet visible
  - The algo would get fewer/no fills — but it might silently succeed with an incorrect model

The insidious version is when someone accidentally sets `publication_offset` to a future-pointing timedelta that lets the algo see tomorrow's actual prices. The results look amazing, but it's entirely fictitious.

### Demonstrating the effect

Let's run the same `ForecastAlgo` but with `publication_offset=timedelta(hours=0)` (no look-ahead protection). In this case, at D-1 12:00 UTC, the delivery period timestamps for day D are all in the future and won't be visible — resulting in `SignalError` on every call and no fills.

In [ ]:
import warnings

# --- Run 1: Correct — publication_offset=36h (forecast published 36h before delivery) ---
forecast_correct = CsvSignalProvider(
    name="price_forecast",
    path=Path("../tests/fixtures/signals/price_forecast.csv"),
    unit="EUR/MWh",
    description="Forecast (correct offset)",
    frequency=timedelta(minutes=15),
    publication_offset=timedelta(hours=36),   # forecasts for D are visible at D-1 12:00 ✓
)

result_correct = BacktestEngine(
    algo=ForecastAlgo(),
    exchange="nordpool",
    start=date(2026, 3, 1),
    end=date(2026, 3, 31),
    products=["NO1_DA"],
    data_dir=Path("../tests/fixtures"),
    capital=Decimal("100000"),
    signals=[forecast_correct],
).run()

# --- Run 2: No offset — signals for delivery day D won't be visible at D-1 12:00 ---
# The CsvSignalProvider logs a warning when publication_offset is None.
# We suppress it here for cleaner output, but in production you want to see it!
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    forecast_no_offset = CsvSignalProvider(
        name="price_forecast",
        path=Path("../tests/fixtures/signals/price_forecast.csv"),
        unit="EUR/MWh",
        description="Forecast (no offset — likely broken)",
        frequency=timedelta(minutes=15),
        publication_offset=None,   # no offset: values only visible at their delivery timestamp
    )

result_no_offset = BacktestEngine(
    algo=ForecastAlgo(),
    exchange="nordpool",
    start=date(2026, 3, 1),
    end=date(2026, 3, 31),
    products=["NO1_DA"],
    data_dir=Path("../tests/fixtures"),
    capital=Decimal("100000"),
    signals=[forecast_no_offset],
).run()

# --- Compare results ---
print("=== publication_offset comparison ===\n")
print(f"With offset=36h:")
print(f"  Total fills:    {len(result_correct.fills)}")
print(f"  Buy win rate:   {result_correct.pnl.buys.win_rate:.1%}")
print(f"  Total alpha:    {result_correct.pnl.total_alpha_eur:+.2f} EUR\n")

print(f"With offset=None (no protection):")
print(f"  Total fills:    {len(result_no_offset.fills)}")
print(f"  Buy win rate:   {result_no_offset.pnl.buys.win_rate:.1%}")
print(f"  Total alpha:    {result_no_offset.pnl.total_alpha_eur:+.2f} EUR\n")

print("Explanation:")
print("  With offset=None, the auction clock is D-1 12:00 UTC.")
print("  Delivery timestamps are D 00:00..23:45 UTC — all in the future.")
print("  get_signal() returns the LAST value with timestamp <= D-1 12:00.")
print("  That is yesterday's last known forecast, which is stale — the algo")
print("  makes decisions based on wrong data. Fewer fills, degraded alpha.")
print()
print("  The correct offset of 36h makes the full day's forecasts visible")
print("  at the time you need them (D-1 12:00 UTC).")

### publication_offset semantics — a precise reference

This is worth understanding exactly, because getting it wrong is easy:

```
publication_offset is POSITIVE and means:
  "This forecast value (for delivery at timestamp T) was published at T - offset."

At simulated time S, the visible values are those where:
  T <= S + offset

Rearranged:
  T - offset <= S
  i.e., publication time <= current simulated time  ← the correct causal check

Example with offset = timedelta(hours=36):
  DA auction clock S = 2026-03-01 12:00 UTC  (deciding on D=2026-03-02)
  Values visible where T <= 2026-03-01 12:00 + 36h = 2026-03-03 00:00 UTC
  The 96 delivery periods of 2026-03-02 (00:00..23:45 UTC) are all <= 2026-03-03 00:00 ✓
  The first period of 2026-03-03 (00:00 UTC) is exactly at the boundary → visible ✓
  Periods of 2026-03-04 are NOT yet visible ✓
```

> **Rule of thumb:** For DA forecasts published the day before, use `publication_offset=timedelta(hours=36)`. This represents "the forecast for day D was published 36 hours before midnight of day D", which is consistent with a noon D-1 publication time.

## 13. Comparing Algos Side by Side

Let's compare all three runs: AlwaysBuy (baseline), ForecastAlgo (correct offset), and ForecastAlgo (no offset).

In [ ]:
runs = [
    ("AlwaysBuy (baseline)", result_always_buy),
    ("ForecastAlgo (offset=36h)", result_forecast),
    ("ForecastAlgo (offset=None)", result_no_offset),
]

print(f"{'Algo':<30} {'Fills':>7} {'Volume(MWh)':>12} {'AvgPrice':>10} {'WinRate':>9} {'Alpha(EUR)':>12}")
print("-" * 85)

for name, res in runs:
    b = res.pnl.buys
    print(
        f"{name:<30} "
        f"{b.count:>7d} "
        f"{float(b.volume_mwh):>12.1f} "
        f"{float(b.avg_price):>10.2f} "
        f"{b.win_rate:>9.1%} "
        f"{float(res.pnl.total_alpha_eur):>+12.2f}"
    )

print(f"\nMarket VWAP: {float(result_always_buy.pnl.market_vwap):.4f} EUR/MWh")
print()
print("Interpretation:")
print("  AlwaysBuy:  buys everything, ~50% win rate, near-zero alpha (expected baseline)")
print("  Forecast:   fewer fills, selectively buys cheap periods → higher win rate & positive alpha")
print("  No offset:  uses stale/wrong forecast values → degraded alpha")

## 14. Visualisations

Let's build a set of charts to visualise the ForecastAlgo's execution quality across the month.

In [ ]:
# Build DataFrames for ForecastAlgo fills and the DA price series
forecast_fills_df = pd.DataFrame([
    {
        "timestamp": f.timestamp,
        "price": float(f.price),
        "volume": float(f.volume),
        "side": str(f.side),
    }
    for f in result_forecast.fills
])

vwap_f = float(result_forecast.pnl.market_vwap)

fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# ─── Panel 1: DA clearing prices with ForecastAlgo fill overlay ───────────────
ax1 = axes[0]
ax1.plot(da_prices["timestamp"], da_prices["price_eur_mwh"],
         linewidth=0.5, color="#94a3b8", alpha=0.9, label="DA clearing price", zorder=1)

if not forecast_fills_df.empty:
    below = forecast_fills_df[forecast_fills_df["price"] < vwap_f]
    above = forecast_fills_df[forecast_fills_df["price"] >= vwap_f]
    ax1.scatter(below["timestamp"], below["price"],
                s=10, color="#16a34a", alpha=0.7, zorder=3,
                label=f"Fill below VWAP ({len(below)})")
    ax1.scatter(above["timestamp"], above["price"],
                s=10, color="#dc2626", alpha=0.7, zorder=3,
                label=f"Fill above VWAP ({len(above)})")

ax1.axhline(vwap_f, color="#1d4ed8", linewidth=1.5, linestyle="--",
            label=f"Market VWAP = {vwap_f:.2f} EUR/MWh", zorder=2)
ax1.set_title("ForecastAlgo — Fill Prices Overlaid on DA Clearing Prices", fontsize=12)
ax1.set_ylabel("EUR/MWh")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax1.legend(fontsize=9, loc="upper right")
ax1.grid(True, alpha=0.2)

# ─── Panel 2: Daily alpha bar chart ───────────────────────────────────────────
ax2 = axes[1]

if not forecast_fills_df.empty:
    # Compute daily alpha: for each fill, alpha = vwap - price (positive = bought cheap)
    forecast_fills_df["date"] = pd.to_datetime(forecast_fills_df["timestamp"]).dt.date
    forecast_fills_df["alpha_per_mwh"] = vwap_f - forecast_fills_df["price"]
    # MWh = MW * 0.25h per 15-min MTU
    forecast_fills_df["alpha_eur"] = forecast_fills_df["alpha_per_mwh"] * forecast_fills_df["volume"] * 0.25

    daily_alpha = forecast_fills_df.groupby("date")["alpha_eur"].sum()

    colors = ["#16a34a" if v >= 0 else "#dc2626" for v in daily_alpha.values]
    ax2.bar(range(len(daily_alpha)), daily_alpha.values, color=colors, alpha=0.8)
    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_xticks(range(len(daily_alpha)))
    ax2.set_xticklabels([str(d) for d in daily_alpha.index], rotation=45, ha="right", fontsize=7)
    ax2.set_title("ForecastAlgo — Daily Alpha vs VWAP (EUR)", fontsize=12)
    ax2.set_ylabel("Alpha (EUR)")
    ax2.grid(True, alpha=0.2, axis="y")
    cumulative = daily_alpha.cumsum().sum()
    ax2.set_xlabel(f"Total cumulative alpha = {float(result_forecast.pnl.total_alpha_eur):+.2f} EUR")

# ─── Panel 3: Win rate comparison bar chart ────────────────────────────────────
ax3 = axes[2]
algo_names = ["AlwaysBuy\n(baseline)", "ForecastAlgo\n(offset=36h)", "ForecastAlgo\n(no offset)"]
win_rates = [
    result_always_buy.pnl.buys.win_rate * 100,
    result_forecast.pnl.buys.win_rate * 100,
    result_no_offset.pnl.buys.win_rate * 100,
]
bar_colors = ["#94a3b8", "#16a34a", "#f59e0b"]
bars = ax3.bar(algo_names, win_rates, color=bar_colors, alpha=0.85, width=0.5)
ax3.axhline(50, color="#dc2626", linewidth=1.5, linestyle="--", label="50% (random baseline)")
ax3.set_ylim(0, 100)
ax3.set_ylabel("Win Rate (%)")
ax3.set_title("Buy Win Rate Comparison (% of fills below VWAP)", fontsize=12)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.2, axis="y")
for bar, val in zip(bars, win_rates):
    ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1.5,
             f"{val:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

plt.tight_layout(pad=2.0)
plt.savefig("forecast_algo_analysis.png", dpi=120, bbox_inches="tight")
plt.show()
print("Charts saved to forecast_algo_analysis.png")

## 15. Inspecting Individual Fills

The `result.fills` tuple gives you access to every single fill for further analysis. Each `Fill` object has: `order_id`, `product_id`, `side`, `price`, `volume`, `timestamp`.

In [ ]:
# Show the first 5 fills from the ForecastAlgo run
print("First 5 fills from ForecastAlgo (offset=36h):")
print(f"{'product_id':<25} {'side':<6} {'price':>10} {'volume':>8} {'timestamp'}")
print("-" * 80)
for fill in result_forecast.fills[:5]:
    print(
        f"{fill.product_id:<25} "
        f"{fill.side:<6} "
        f"{float(fill.price):>10.2f} "
        f"{float(fill.volume):>8.2f} "
        f"{fill.timestamp.isoformat()}"
    )

# Count fills by delivery period time-of-day to see intraday pattern
if result_forecast.fills:
    print(f"\nTotal fills: {len(result_forecast.fills)}")
    print(f"Fill rate:   {len(result_forecast.fills)} / 2976 MTUs = "
          f"{len(result_forecast.fills)/2976:.1%} of all periods traded")

# Show price distribution of fills
prices = [float(f.price) for f in result_forecast.fills]
if prices:
    print(f"\nFill price statistics:")
    print(f"  Min:    {min(prices):.2f} EUR/MWh")
    print(f"  Max:    {max(prices):.2f} EUR/MWh")
    print(f"  Mean:   {sum(prices)/len(prices):.2f} EUR/MWh")
    print(f"  VWAP:   {float(result_forecast.pnl.market_vwap):.2f} EUR/MWh")

## 16. Auto-Discovery of Signal CSV Files

In the examples above we passed the `CsvSignalProvider` explicitly to `BacktestEngine`. There is a more convenient shortcut: **auto-discovery**.

If your algo subscribes to a signal named `"my_signal"` and you do not pass an explicit provider, the engine looks for:

```
{data_dir}/signals/my_signal.csv
```

If it finds the file, it auto-loads it as a `CsvSignalProvider` with **no publication_offset** (actuals mode). This is convenient for quickly loading historical actuals, but you should always set an explicit `publication_offset` for forecast data to prevent look-ahead bias.

```python
# Auto-discovery: engine will look for tests/fixtures/signals/price_forecast.csv
engine = BacktestEngine(
    algo=ForecastAlgo(),   # ForecastAlgo calls subscribe_signal("price_forecast")
    exchange="nordpool",
    start=date(2026, 3, 1),
    end=date(2026, 3, 31),
    products=["NO1_DA"],
    data_dir=Path("../tests/fixtures"),
    capital=Decimal("100000"),
    # No `signals=` argument — engine auto-discovers from tests/fixtures/signals/
)
```

> **Warning:** Auto-discovered providers have `publication_offset=None`, which means values are visible at their exact timestamp. This is suitable for actuals (observed historical data) but will silently introduce look-ahead bias for any forecast data. Always pass an explicit `CsvSignalProvider` with the correct `publication_offset` for forecast signals.

## 17. Quick Reference: Key Types

Here is a concise reference for the types you will use most often when building algos.

In [ ]:
from nexa_backtest.types import Order, Fill, Position, AuctionInfo, MTU, Side
from nexa_backtest.context import SignalValue
from decimal import Decimal
from datetime import datetime, timezone

# ─── Order construction ────────────────────────────────────────────────────────
print("=== Order ===")
buy_order = Order.buy("NO1-QH-0900", volume_mw=5.0, price_eur_mwh=48.50)
print(f"  Buy order:   product={buy_order.product_id}, volume={buy_order.volume_mw} MW, "
      f"price={buy_order.price_eur_mwh} EUR/MWh")

sell_order = Order.sell("NO1-QH-0900", volume_mw=5.0, price_eur_mwh=52.00)
print(f"  Sell order:  product={sell_order.product_id}, volume={sell_order.volume_mw} MW, "
      f"price={sell_order.price_eur_mwh} EUR/MWh")

market_order = Order.market("NO1-QH-0900", side=Side.BUY, volume_mw=2.0)
print(f"  Market order: side={market_order.side}, price_eur_mwh={market_order.price_eur_mwh} (None = always fills)")

# ─── AuctionInfo fields available in on_auction_open ──────────────────────────
print("\n=== AuctionInfo (passed to on_auction_open) ===")
auction = AuctionInfo(
    product_id="NO1-QH-0900",
    auction_type="DA",
    gate_closure_time=datetime(2026, 3, 1, 13, 0, tzinfo=timezone.utc),
    zone="NO1",
)
print(f"  product_id:        {auction.product_id}")
print(f"  auction_type:      {auction.auction_type}  ('DA' = Day-Ahead)")
print(f"  gate_closure_time: {auction.gate_closure_time.isoformat()}")
print(f"  zone:              {auction.zone}")

# ─── SignalValue fields available from ctx.get_signal() ───────────────────────
print("\n=== SignalValue (returned by ctx.get_signal()) ===")
sv = SignalValue(
    name="price_forecast",
    timestamp=datetime(2026, 3, 2, 9, 0, tzinfo=timezone.utc),
    value=47.32,
)
print(f"  name:      {sv.name}")
print(f"  timestamp: {sv.timestamp.isoformat()}  (the delivery period this value is for)")
print(f"  value:     {sv.value}  (units depend on the signal — here EUR/MWh)")

## 18. What Comes Next

You have now completed the core nexa-backtest walkthrough. Here is what is coming in future stages of the framework and what to explore next.

### Stage 2: Intraday Continuous (IDC) + Windowed Replay

The Day-Ahead market is only part of the picture. Intraday continuous trading (Nord Pool's XBID market, EPEX SPOT CID) runs all day and is matched using price-time priority against a live order book. Key additions:

- **IDC matching engine** with price-time priority and order book replay
- **Windowed data loading** — IDC data is ~10 GB/year/zone, so it uses a sliding window loaded from Parquet row groups to keep memory under 500 MB
- **`@algo` decorator** — an async event-stream interface for more complex strategies
- **EPEX SPOT and EEX adapters** — same principles, different gate closure rules and product naming

### Stage 3: ML Models + Validation

- **Model registry** — register ONNX or scikit-learn models and call `ctx.predict()` from your algo
- **Full validation pipeline** — `nexa validate` catches look-ahead bias, resource safety issues, and type errors before you run
- **Multi-algo replay** — run many algos against the same data window in a single pass

### Stage 4: Paper Trading and Live Trading

- **Paper engine** — real-time clock, live market data, simulated fills
- **Live engine** — wraps `nexa-connect` for real order submission to exchanges
- **Code protection** — compile algos to Cython or Nuitka for IP protection

### Extending the examples in this notebook

Some immediate experiments you can try:

1. **Change the threshold logic**: instead of mean forecast, try buying only the cheapest quartile of MTUs
2. **Add a sell strategy**: mirror the buy logic — sell when the forecast is above average
3. **Test different date ranges**: create synthetic fixtures for different months and see how the algo performs
4. **Add multiple signals**: subscribe to a second signal (e.g., a load forecast) and combine them

### Useful reference

```python
# Full import reference for algo code
from nexa_backtest import SimpleAlgo, BacktestEngine, CsvSignalProvider
from nexa_backtest import Order, Fill, AuctionInfo, Side
from nexa_backtest.context import TradingContext, SignalValue
from nexa_backtest.exceptions import SignalError, DataError
from datetime import date, timedelta
from decimal import Decimal
from pathlib import Path
```

## 19. Summary

Here is everything covered in this walkthrough:

| Topic | Key point |
|---|---|
| **What a backtester is** | Replays historical data through your algo and returns fills + PnL metrics |
| **Why power markets need their own** | Auctions, MTU resolution, gate closure, and MWh delivery differ from equities |
| **MTU** | 15-minute delivery period, 96 per day |
| **DA auction** | Bids submitted before gate closure (noon D-1), matched at a single clearing price |
| **Price-taker assumption** | Your bid doesn't move the clearing price; fill at clearing if bid is good enough |
| **Matching rule (buy)** | Fill if `bid_price >= clearing_price`; fill at `clearing_price` |
| **Matching rule (sell)** | Fill if `bid_price <= clearing_price`; fill at `clearing_price` |
| **VWAP** | Volume-weighted average of all clearing prices; the benchmark |
| **Alpha** | Execution quality vs VWAP; positive = beat the benchmark |
| **Win rate** | Fraction of fills on the favourable side of VWAP |
| **SimpleAlgo lifecycle** | `on_setup` → `on_signal` (per day) → `on_auction_open` (per MTU) → `on_fill` → `on_teardown` |
| **TradingContext** | The only interface between your algo and the engine; same API in backtest/paper/live |
| **Signals** | Time-series data injected into the algo; respects publication_offset |
| **publication_offset** | Positive timedelta: value for delivery T visible when clock >= T - offset |
| **Look-ahead bias** | Using future data in a decision; silently inflates backtest performance |
| **CsvSignalProvider** | Loads CSV as a signal; set publication_offset for any forecast data |